# Notebook 09: Crossover Analysis

**One Sensor, One Year — Edition 1: India Breathes**

The brief asks: on how many days did renewables beat nuclear? Beat hydro? When did wind+solar exceed gas? These crossover moments are milestones — even if they're not widely reported.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df24 = pd.read_csv('../data/processed/india_2024_derived.csv', parse_dates=['date'])
df24['wind_solar'] = df24['wind'].fillna(0) + df24['solar'].fillna(0)

palette = {
    'coal': '#3D2B1F', 'lignite': '#6B4226', 'gas': '#4A90D9',
    'nuclear': '#2EC4B6', 'hydro': '#1B4F72', 'wind': '#85C1E9',
    'solar': '#F5B041', 'other_re': '#A569BD', 'renewables': '#27AE60',
    'wind_solar': '#E67E22',
}

## 1. Crossover Scoreboard

In [2]:
crossovers = [
    ('renewables', 'nuclear', 'Renewables > Nuclear'),
    ('renewables', 'hydro', 'Renewables > Hydro'),
    ('wind_solar', 'nuclear', 'Wind+Solar > Nuclear'),
    ('wind_solar', 'gas', 'Wind+Solar > Gas'),
    ('wind_solar', 'hydro', 'Wind+Solar > Hydro'),
    ('solar', 'nuclear', 'Solar > Nuclear'),
    ('solar', 'wind', 'Solar > Wind'),
    ('solar', 'hydro', 'Solar > Hydro'),
    ('wind', 'nuclear', 'Wind > Nuclear'),
    ('wind', 'gas', 'Wind > Gas'),
    ('hydro', 'nuclear', 'Hydro > Nuclear'),
]

print('CROSSOVER SCOREBOARD — India Grid 2024')
print('=' * 70)
print(f'{"Matchup":30s} {"Days":>6s} {"of 366":>7s} {"First":>12s} {"Last":>12s}')
print('-' * 70)

for a, b, label in crossovers:
    mask = df24[a] > df24[b]
    days = mask.sum()
    pct = days / len(df24) * 100
    if days > 0:
        first = df24[mask]['date'].min().strftime('%b %d')
        last = df24[mask]['date'].max().strftime('%b %d')
    else:
        first = last = 'never'
    print(f'{label:30s} {days:6d} {pct:6.1f}% {first:>12s} {last:>12s}')

CROSSOVER SCOREBOARD — India Grid 2024
Matchup                          Days  of 366        First         Last
----------------------------------------------------------------------
Renewables > Nuclear              366  100.0%       Jan 01       Dec 31
Renewables > Hydro                318   86.9%       Jan 01       Dec 31
Wind+Solar > Nuclear              366  100.0%       Jan 01       Dec 31
Wind+Solar > Gas                  366  100.0%       Jan 01       Dec 31
Wind+Solar > Hydro                303   82.8%       Jan 01       Dec 31
Solar > Nuclear                   366  100.0%       Jan 01       Dec 31
Solar > Wind                      287   78.4%       Jan 01       Dec 31
Solar > Hydro                     194   53.0%       Jan 01       Dec 31
Wind > Nuclear                    236   64.5%       Jan 06       Dec 31
Wind > Gas                        321   87.7%       Jan 01       Dec 31
Hydro > Nuclear                   366  100.0%       Jan 01       Dec 31


## 2. Renewables vs Nuclear — Daily Head-to-Head

In [3]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df24['date'], y=df24['renewables'],
    name='Renewables (W+S+Other)', mode='lines',
    line=dict(width=1.5, color=palette['renewables']),
    hovertemplate='RE: %{y:.0f} MU<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=df24['date'], y=df24['nuclear'],
    name='Nuclear', mode='lines',
    line=dict(width=1.5, color=palette['nuclear']),
    hovertemplate='Nuclear: %{y:.0f} MU<extra></extra>',
))

fig.update_layout(
    title='Renewables vs Nuclear — Daily Generation (India 2024)',
    yaxis_title='Generation (MU/day)',
    hovermode='x unified',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=400,
)
fig.show()

re_wins = (df24['renewables'] > df24['nuclear']).sum()
print(f'Renewables beat Nuclear on {re_wins} of 366 days ({re_wins/366*100:.0f}%)')
print(f'Renewables avg: {df24["renewables"].mean():.0f} MU vs Nuclear avg: {df24["nuclear"].mean():.0f} MU')
print(f'Ratio: Renewables are {df24["renewables"].mean()/df24["nuclear"].mean():.1f}x Nuclear on average')

Renewables beat Nuclear on 366 of 366 days (100%)
Renewables avg: 583 MU vs Nuclear avg: 148 MU
Ratio: Renewables are 3.9x Nuclear on average


## 3. Wind+Solar vs Gas — The Displacement Story

In [4]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df24['date'], y=df24['wind_solar'],
    name='Wind+Solar', mode='lines',
    line=dict(width=1.5, color=palette['wind_solar']),
    hovertemplate='W+S: %{y:.0f} MU<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=df24['date'], y=df24['gas'],
    name='Gas', mode='lines',
    line=dict(width=1.5, color=palette['gas']),
    hovertemplate='Gas: %{y:.0f} MU<extra></extra>',
))

fig.update_layout(
    title='Wind+Solar vs Gas — Daily Generation (India 2024)',
    yaxis_title='Generation (MU/day)',
    hovermode='x unified',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=400,
)
fig.show()

ws_beats_gas = (df24['wind_solar'] > df24['gas']).sum()
print(f'Wind+Solar beat Gas on {ws_beats_gas} of 366 days ({ws_beats_gas/366*100:.0f}%)')
print(f'W+S avg: {df24["wind_solar"].mean():.0f} MU vs Gas avg: {df24["gas"].mean():.0f} MU')
print(f'Ratio: Wind+Solar are {df24["wind_solar"].mean()/df24["gas"].mean():.1f}x Gas on average')

Wind+Solar beat Gas on 366 of 366 days (100%)
W+S avg: 556 MU vs Gas avg: 93 MU
Ratio: Wind+Solar are 6.0x Gas on average


## 4. Solar vs Wind — The Internal Race

In [5]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df24['date'], y=df24['solar'],
    name='Solar', mode='lines',
    line=dict(width=1.5, color=palette['solar']),
    hovertemplate='Solar: %{y:.0f} MU<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=df24['date'], y=df24['wind'].fillna(0),
    name='Wind', mode='lines',
    line=dict(width=1.5, color=palette['wind']),
    hovertemplate='Wind: %{y:.0f} MU<extra></extra>',
))

fig.update_layout(
    title='Solar vs Wind — Daily Generation (India 2024)',
    yaxis_title='Generation (MU/day)',
    hovermode='x unified',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=400,
)
fig.show()

solar_wins = (df24['solar'] > df24['wind'].fillna(0)).sum()
print(f'Solar beat Wind on {solar_wins} of 366 days ({solar_wins/366*100:.0f}%)')
print(f'Solar wins outside monsoon, Wind wins during monsoon.')
print(f'They\'re complementary — one peaks when the other dips.')

Solar beat Wind on 288 of 366 days (79%)
Solar wins outside monsoon, Wind wins during monsoon.
They're complementary — one peaks when the other dips.


## 5. Cumulative Crossover Count Through the Year

In [6]:
key_crossovers = [
    ('renewables', 'nuclear', 'RE > Nuclear', '#27AE60'),
    ('wind_solar', 'gas', 'W+S > Gas', '#E67E22'),
    ('solar', 'wind', 'Solar > Wind', '#F5B041'),
    ('solar', 'nuclear', 'Solar > Nuclear', '#D4AC0D'),
    ('wind', 'nuclear', 'Wind > Nuclear', '#85C1E9'),
]

fig = go.Figure()

for a, b, label, color in key_crossovers:
    cumcount = (df24[a] > df24[b]).cumsum()
    fig.add_trace(go.Scatter(
        x=df24['date'], y=cumcount,
        name=label, mode='lines',
        line=dict(width=2, color=color),
        hovertemplate=f'{label}: %{{y}} days<extra></extra>',
    ))

# Reference line: every day
fig.add_trace(go.Scatter(
    x=df24['date'], y=np.arange(1, 367),
    name='Every day', mode='lines',
    line=dict(width=1, color='grey', dash='dot'),
    hoverinfo='skip',
))

fig.update_layout(
    title='Cumulative Crossover Days Through 2024',
    yaxis_title='Cumulative days where A > B',
    hovermode='x unified',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=500,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0.5, xanchor='center'),
)
fig.show()

print('Dotted grey line = every day. Lines close to it mean the crossover happens almost daily.')

Dotted grey line = every day. Lines close to it mean the crossover happens almost daily.


## 6. The Gap Chart — How Far Apart Are They?

In [7]:
fig = make_subplots(rows=2, cols=1, subplot_titles=[
    'Wind+Solar minus Gas (positive = W+S wins)',
    'Wind+Solar minus Nuclear (positive = W+S wins)'
], vertical_spacing=0.12)

for i, (col_name, label) in enumerate([('gas', 'Gas'), ('nuclear', 'Nuclear')], 1):
    gap = df24['wind_solar'] - df24[col_name]
    colors = ['#27AE60' if v > 0 else '#C0392B' for v in gap]
    
    fig.add_trace(go.Bar(
        x=df24['date'], y=gap,
        marker_color=colors,
        showlegend=False,
        hovertemplate='Gap: %{y:.0f} MU | %{x|%b %d}<extra></extra>',
    ), row=i, col=1)
    fig.update_yaxes(title_text='MU gap', row=i, col=1)

fig.update_layout(
    title='Wind+Solar Surplus/Deficit vs Gas and Nuclear',
    height=600, plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
)
fig.show()

## Key Findings

1. **Renewables beat Nuclear every single day** — it's not even close. RE is 3-4x Nuclear daily.
2. **Wind+Solar beat Gas on almost every day** — gas is a shrinking player in India's grid.
3. **Solar beats Wind most of the year** — except during monsoon when wind surges past solar. They're beautifully complementary.
4. **Solar beats Nuclear on most days** — solar alone outproduces India's entire nuclear fleet.
5. **The cumulative chart tells the acceleration story** — crossover lines that hug the "every day" reference line mean those milestones are now routine, not exceptional.
6. **For the essay:** These crossovers are narrative gold. "On X days in 2024, the sun and wind alone produced more electricity than all of India's nuclear plants combined."

→ Next: Notebook 10 — Art Prototypes